# Infra Quick Reference: Docker, K8s & Friends

MLEs are expected to know enough Docker and Kubernetes to deploy and debug their own models. This note covers interview-level (not practitioner-level) Docker and K8s knowledge — what to say when asked how you'd deploy a model.

## What Interviewers Test
- What goes in a Dockerfile for training vs serving
- K8s core objects: Pod, Deployment, Service, HPA — what each does
- Spot instance handling: checkpointing and preemption
- Cloud storage patterns for large datasets
- Secrets and config management

## Dockerfile Patterns

**Training Dockerfile:**
```dockerfile
# Training: large image, GPU support, all libraries
FROM pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime

WORKDIR /app

# Install dependencies first (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy source code (changes frequently; separate layer)
COPY src/ src/
COPY configs/ configs/

# Data comes in at runtime (mounted volume or cloud storage)
# Never COPY training data into the image

ENTRYPOINT ["python", "src/train.py"]
# CMD overrides args: docker run ... --epochs 10 --lr 0.001
```

**Serving Dockerfile:**
```dockerfile
# Serving: lean image, CPU inference, minimal footprint
FROM python:3.11-slim

WORKDIR /app
COPY requirements_serve.txt .
RUN pip install --no-cache-dir -r requirements_serve.txt

# Copy model artifact (or mount at runtime)
COPY model_artifacts/ model_artifacts/
COPY src/serve.py src/serve.py

EXPOSE 8080
# Health check (K8s liveness probe)
HEALTHCHECK CMD curl -f http://localhost:8080/health || exit 1

CMD ["python", "src/serve.py", "--port", "8080"]
```

> 💡 **Interview Tip:** Separate training and serving images. Training is large (PyTorch + CUDA + all of sklearn); serving should be minimal for fast cold-start and reduced attack surface. Never put secrets or credentials in a Dockerfile — use environment variables or secret managers.


## Kubernetes Core Objects for MLEs

| Object | What it does | MLE touches it for |
|---|---|---|
| **Pod** | Smallest deployable unit; one or more containers | Rarely directly; managed by Deployment |
| **Deployment** | Manages replicas of a Pod; handles rolling updates | Deploy a new model version |
| **Service** | Stable network endpoint for a Deployment | Expose model server to application layer |
| **HPA** | Horizontal Pod Autoscaler — scale replicas on CPU/QPS | Auto-scale serving under load |
| **ConfigMap** | Key-value config injected at runtime | Model hyperparams, feature configs |
| **Secret** | Encrypted ConfigMap for sensitive values | API keys, DB passwords |
| **Job/CronJob** | Run-to-completion tasks | Batch training, data processing |
| **PersistentVolumeClaim** | Persistent storage | Checkpoints, datasets |

```yaml
# Minimal serving Deployment (reference — not runnable here)
# apiVersion: apps/v1
# kind: Deployment
# spec:
#   replicas: 3           # Start with 3 replicas
#   selector: { matchLabels: { app: model-server } }
#   template:
#     spec:
#       containers:
#       - name: model-server
#         image: my-model:v2.1
#         resources:
#           requests: { memory: "2Gi", cpu: "1" }
#           limits:   { memory: "4Gi", cpu: "2" }
#         env:
#         - name: MODEL_VERSION
#           value: "v2.1"
```


In [ ]:
# Simulate spot instance preemption handling
import time, json, os, signal

class TrainingCheckpointer:
    """Handles graceful checkpoint on SIGTERM (spot preemption)."""
    
    def __init__(self, checkpoint_dir='/tmp/checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.step = 0
        self.best_metric = float('inf')
        signal.signal(signal.SIGTERM, self._handle_sigterm)  # register preemption handler
    
    def _handle_sigterm(self, signum, frame):
        print(f"SIGTERM received at step {self.step} — saving emergency checkpoint")
        self.save(self.step, emergency=True)
        exit(0)
    
    def save(self, step, metric=None, emergency=False):
        checkpoint = {
            'step': step,
            'metric': metric,
            'emergency': emergency,
            'timestamp': time.time(),
        }
        # In production: save model weights + optimizer state here
        path = os.path.join(self.checkpoint_dir,
                            f"{'emergency' if emergency else 'checkpoint'}_step{step}.json")
        with open(path, 'w') as f:
            json.dump(checkpoint, f)
        print(f"Checkpoint saved: {path}")
        return path
    
    def load_latest(self):
        files = sorted([f for f in os.listdir(self.checkpoint_dir) if f.endswith('.json')])
        if not files:
            return None
        latest = json.load(open(os.path.join(self.checkpoint_dir, files[-1])))
        print(f"Resuming from step {latest['step']}")
        return latest

# Demonstrate usage
checkpointer = TrainingCheckpointer()

# Simulate training with periodic checkpoints
for step in range(0, 100, 25):
    checkpointer.step = step
    if step % 50 == 0:
        checkpointer.save(step, metric=1.0 - step/200)

state = checkpointer.load_latest()
print(f"Would resume from step {state['step']}")


## Cloud Storage Patterns for ML

| Use case | Pattern | Notes |
|---|---|---|
| **Training data** | S3/GCS bucket, partitioned by date/version | Never copy to local disk; stream or pre-cache |
| **Model artifacts** | S3/GCS + model registry | Versioned, immutable paths |
| **Feature store offline** | Parquet on S3/GCS, partitioned by date | Point-in-time reads via date prefix |
| **Checkpoints** | S3/GCS bucket, per-run prefix | Delete old checkpoints to save cost |
| **Logs** | CloudWatch / GCS logs | Structured JSON for queryability |

**Naming convention for reproducibility:**
```
s3://company-ml/
  experiments/<experiment_id>/<run_id>/
    checkpoints/
    artifacts/
    logs/
  datasets/<name>/<version>/
  models/<name>/<version>/
```


## Secrets & Config Management

**Never:**
- Hardcode API keys in code
- Put secrets in Docker images
- Commit secrets to git

**Do:**
- Inject secrets via environment variables from a secrets manager (AWS Secrets Manager, GCP Secret Manager, K8s Secrets)
- Use ConfigMaps for non-sensitive config (hyperparameters, feature lists)
- Rotate secrets on a schedule; alert on unauthorized access

```python
# Good: read from environment at runtime
import os
API_KEY = os.environ.get('OPENAI_API_KEY')   # injected by K8s Secret or .env file
if not API_KEY:
    raise ValueError("OPENAI_API_KEY not set")

# Bad: hardcoded
# API_KEY = "sk-abc123..."
```


## Common Interview Questions

**Q: Why separate training and serving Docker images?**
Training needs GPU drivers, large ML frameworks (PyTorch + CUDA), and all development libraries — often several GBs. Serving needs only inference dependencies, which can be under 200MB for simple models. Smaller serving images mean faster cold-start, reduced attack surface, and lower registry storage costs. Use multi-stage builds if sharing code.

**Q: What does the K8s HPA do and how would you use it for ML serving?**
The Horizontal Pod Autoscaler scales the number of pod replicas based on a metric (CPU utilization, custom QPS metric). For ML serving: set a target CPU utilization (e.g., 60%) and HPA will add/remove replicas as traffic fluctuates. For GPU serving, use a custom metric (requests per second per GPU) since CPU isn't the bottleneck.

**Q: How do you handle spot instance preemption during training?**
Register a SIGTERM handler that triggers an emergency checkpoint when preemption is signaled (cloud providers give 2-minute warning). Checkpoint periodically (every N steps) so at most N steps of work are lost on preemption. Use a retry-enabled job scheduler (like Kubernetes Job or SageMaker training) that auto-restarts on the next available instance and resumes from the latest checkpoint.

**Q: What's the difference between a K8s ConfigMap and Secret?**
Both inject configuration into Pods. ConfigMaps are for non-sensitive config (feature lists, hyperparams) stored in plaintext. Secrets are for sensitive values (passwords, API keys) stored base64-encoded (not encrypted by default in K8s, though can be encrypted at rest). In production, use external secret managers (AWS Secrets Manager) and only surface values as environment variables in the Pod.

## Key Takeaways
- Separate training (large, GPU) and serving (minimal, CPU) Docker images
- K8s essentials: Deployment (replicas + rolling updates), Service (stable endpoint), HPA (autoscale), Job (one-off tasks)
- Spot preemption: register SIGTERM handler → emergency checkpoint → job retries from checkpoint
- Cloud storage: partition by date/version; versioned artifact paths for reproducibility
- Secrets: never in code or images; inject via environment variables from a secrets manager
- Config: ConfigMap for non-sensitive; Secret (or external manager) for credentials